# 03 - Data Preparation (CRISP-DM phase 3)

**Goal:** produce ONE clean merged table, reproducible by running this notebook top to bottom, plus a cleaning log.

Sources merged here:
1. **Primary:** `data/raw/Fraud_Data.csv`
2. **Static lookup:** `data/raw/IpAddress_to_Country.csv` (IP ranges -> country)
3. **API:** `data/processed/ip_lookup.csv` (columns: `ip, country, country_code, status, proxy, hosting`)
4. **Scrape:** `data/raw/cpi_raw_table.csv` (Wikipedia Corruption Perceptions Index)

**Rules for this notebook**
- Raw files are never modified. Everything is done on copies.
- Every decision is logged with `log(step, source, did, because, rows)` ("did X because Y").
- Failed lookups and unmatched countries get a **flag**, not a guess, and are not silently dropped.
- `ip_address` is **fractional** in 151,111 of 151,112 rows (found in `01_data_understanding`). It is **floored, never rounded**, and the original value is kept as `ip_address_raw`. This encoding is still unconfirmed by the team.
- Working agreement from `01_data_understanding`: start this notebook only after the team says data understanding is finished.
- Feature engineering (`seconds_since_signup`, `users_per_device`, ...) and the train/test split are **not** here: they belong to `05_features`. Encoding and scaling happen after the split.

## 0. Setup

In [ ]:
import re
import ipaddress
import sqlite3
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

RAW  = Path("../data/raw")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

FRAUD_CSV     = RAW / "Fraud_Data.csv"
STATIC_CSV    = RAW / "IpAddress_to_Country.csv"
IP_LOOKUP_CSV = PROC / "ip_lookup.csv"
CPI_CSV       = RAW / "cpi_raw_table.csv"
DB_PATH       = PROC / "fraud.db"

CPI_YEAR = 2015   # transactions are from 2015; change if the team decides otherwise
COUNTRY_SOURCE = "api"   # which country feeds the CPI join: "api" or "static" (team decision pending)

# ---- cleaning log: every decision as "did X because Y" ----
cleaning_log = []

def log(step, source, did, because, rows=None):
    cleaning_log.append({"step": step, "source": source, "did": did,
                         "because": because, "rows_affected": rows})
    print(f"[{source}] {did}" + (f"  (rows: {rows})" if rows is not None else ""))

## 1. Primary data: `Fraud_Data.csv`
### 1.1 Load (raw copy kept untouched in `raw`)

In [ ]:
raw = pd.read_csv(FRAUD_CSV)
df = raw.copy()
print("raw shape:", raw.shape)
df.head()

### 1.2 Fix types (timestamps are strings in the CSV)

In [ ]:
for c in ["signup_time", "purchase_time"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")

n_bad_dt = int(df[["signup_time", "purchase_time"]].isna().any(axis=1).sum())
log("types", "fraud_data", "parsed signup_time and purchase_time to datetime64",
    "they are stored as strings (object); needed for time features and SQL", len(df))
if n_bad_dt:
    df = df.dropna(subset=["signup_time", "purchase_time"])
    log("types", "fraud_data", "dropped rows with unparseable timestamps",
        "cannot build time features without a valid timestamp", n_bad_dt)
else:
    print("no unparseable timestamps")

df["device_id"] = df["device_id"].astype(str).str.strip()

### 1.3 Duplicates

In [ ]:
n_dup_rows  = int(df.duplicated().sum())
n_dup_users = int(df.duplicated(subset="user_id").sum())
print("duplicate rows:", n_dup_rows, "| duplicate user_id:", n_dup_users)

if n_dup_rows:
    df = df.drop_duplicates()
if n_dup_users:
    df = df.drop_duplicates(subset="user_id", keep="first")
log("duplicates", "fraud_data", "checked full-row and user_id duplicates; dropped any found",
    "a transaction must appear once or it biases the model and the split", n_dup_rows + n_dup_users)

### 1.4 Impossible values
Fractional IP values are **not** treated as errors here: they are in almost every row (see 1.7).

In [ ]:
checks = {
    "purchase_before_signup":      df.purchase_time < df.signup_time,
    "age_outside_18_100":          ~df.age.between(18, 100),
    "purchase_value_le_0":         df.purchase_value <= 0,
    "ip_outside_ipv4_range":       ~df.ip_address.between(0, 2**32 - 1),
}
summary = pd.Series({k: int(v.sum()) for k, v in checks.items()}, name="rows")
display(summary)

bad_mask = np.logical_or.reduce([v.to_numpy() for v in checks.values()])
n_bad = int(bad_mask.sum())
if n_bad:
    df = df[~bad_mask]
log("impossible_values", "fraud_data",
    "checked purchase<signup, age range, value<=0 and IP validity; dropped violations",
    "these rows are data errors, not behaviour (0 means the check passed)", n_bad)

### 1.5 Outliers in `purchase_value` and `age`
Outliers are **reported, not removed**: a large purchase can be exactly the fraud we want to catch.

In [ ]:
for col in ["purchase_value", "age"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[col] < lo) | (df[col] > hi)).sum())
    print(f"{col}: IQR fences [{lo:.1f}, {hi:.1f}] -> {n_out} outliers "
          f"(min={df[col].min()}, max={df[col].max()})")
    log("outliers", "fraud_data", f"checked {col} with the IQR rule and kept all values",
        "values are plausible and extreme purchases may be fraud signal", n_out)

### 1.6 Standardize categoricals (`source`, `browser`, `sex`)

In [ ]:
for c in ["source", "browser"]:
    df[c] = df[c].astype(str).str.strip().str.lower()
df["sex"] = df["sex"].astype(str).str.strip().str.upper()

for c in ["source", "browser", "sex"]:
    print(df[c].value_counts(dropna=False), "\n")

log("categoricals", "fraud_data",
    "stripped whitespace; lower-cased source and browser; upper-cased sex",
    "avoid the same category appearing under different spellings", len(df))

### 1.7 IP address: float -> integer -> dotted-quad
`ip_address` is a float64 with decimals (e.g. `732758368.79972`) in 151,111 of 151,112 rows. **Team assumption (unconfirmed): the whole-number part is the IPv4 number**, so we use `floor`. We never round to the nearest integer, because that could select the next address. The original value is kept as `ip_address_raw`. The dotted string `ip` is the key used to join the API results.

In [ ]:
n_fractional = int((df["ip_address"] % 1 != 0).sum())
print(f"fractional ip_address values: {n_fractional:,} of {len(df):,}")
df["ip_int"] = np.floor(df["ip_address"]).astype("int64")
df["ip"] = df["ip_int"].map(lambda x: str(ipaddress.IPv4Address(int(x))))
df = df.rename(columns={"ip_address": "ip_address_raw"})

# sanity checks
assert df["ip"].str.count(r"\.").eq(3).all()
sample = df.sample(5, random_state=0)
assert all(int(ipaddress.IPv4Address(ip)) == i for ip, i in zip(sample.ip, sample.ip_int))

log("ip", "fraud_data", "floored ip_address (float with decimals) -> int -> dotted-quad string; kept original as ip_address_raw",
    "lookups need whole IPv4 numbers; flooring is the team's unconfirmed assumption (rounding could pick the next address)", n_fractional)
print("unique IPs:", df["ip"].nunique())
df = df.reset_index(drop=True)
n_primary = len(df)
print("rows after primary cleaning:", n_primary)
df[["ip_int", "ip"]].head()

**Data-quality note (primary):** _TODO (team): write 3-5 lines with the real numbers from above (nulls, duplicates, impossible values, outliers, what was really "messy": float IPs, string timestamps, free-text categoricals), the fractional IPs (151,111 of 151,112), and the 7,600 purchases made exactly one second after signup (all labeled fraud; kept, to be investigated in 01)._

## 1b. Static lookup: `IpAddress_to_Country.csv`
IP ranges (`lower_bound_ip_address` .. `upper_bound_ip_address`, inclusive) -> country. Purchases outside every range get `"Unknown"` plus `static_matched = 0` (they are kept). Coverage found in `01_data_understanding` was about 85.5%.

In [ ]:
STATIC_AVAILABLE = STATIC_CSV.exists()
if STATIC_AVAILABLE:
    static = pd.read_csv(STATIC_CSV).sort_values("lower_bound_ip_address").reset_index(drop=True)
    starts = static["lower_bound_ip_address"].to_numpy(dtype="float64")
    ends = static["upper_bound_ip_address"].to_numpy(dtype="float64")
    assert (starts <= ends).all(), "range with lower > upper"
    assert (starts[1:] > ends[:-1]).all(), "overlapping ranges: resolve before matching"

    ip_values = df["ip_int"].to_numpy(dtype="float64")
    pos = np.searchsorted(starts, ip_values, side="right") - 1
    hit = (pos >= 0) & (ip_values <= ends[pos.clip(0)])
    df["ip_country_static"] = np.where(hit, static["country"].to_numpy()[pos.clip(0)], "Unknown")
else:
    print("WARNING: IpAddress_to_Country.csv not found -> ip_country_static = 'Unknown' for all rows.")
    df["ip_country_static"] = "Unknown"

df["static_matched"] = (df["ip_country_static"] != "Unknown").astype(int)
log("static_lookup", "ip_ranges", "matched floored IPs to inclusive ranges; no match -> 'Unknown' + static_matched=0",
    "ranges do not cover every address; rows are kept and flagged", int((df["static_matched"] == 0).sum()))
print("static coverage: %.2f%%" % (100 * df["static_matched"].mean()))

## 2. API data: `ip_lookup.csv` (ip-api.com)
If the file does not exist yet, the notebook runs with an **empty placeholder** so everyone can keep working. In that case every row gets `ip_country = "Unknown"` and the final file is saved with a `_PLACEHOLDER` suffix so it can never be mistaken for the real one.

In [ ]:
LOOKUP_COLS = ["ip", "country", "country_code", "status", "proxy", "hosting"]
USING_PLACEHOLDER = not IP_LOOKUP_CSV.exists()

if USING_PLACEHOLDER:
    print("WARNING: ip_lookup.csv not found -> using an EMPTY placeholder. Do not use this output for modeling.")
    ip_raw = pd.DataFrame(columns=LOOKUP_COLS)
else:
    ip_raw = pd.read_csv(IP_LOOKUP_CSV)

ipl = ip_raw.copy()
ipl.columns = [c.strip().lower() for c in ipl.columns]
for col in LOOKUP_COLS:
    if col not in ipl.columns:
        ipl[col] = np.nan
        log("api", "ip_lookup", f"column '{col}' missing in the response file; created as empty",
            "the response did not include this field", len(ipl))

for col in ["ip", "country", "country_code", "status"]:
    ipl[col] = ipl[col].astype("string").str.strip()
ipl["status"] = ipl["status"].str.lower()

print("rows in lookup:", len(ipl))
print(ipl["status"].value_counts(dropna=False))

In [ ]:
# one row per IP: prefer a successful lookup over a failed one
ipl["lookup_ok"] = (ipl["status"] == "success") & ipl["country"].notna()
before = len(ipl)
ipl = ipl.sort_values("lookup_ok", ascending=False).drop_duplicates("ip", keep="first")
log("api", "ip_lookup", "kept one row per IP, preferring successful lookups",
    "retries may have produced duplicate rows", before - len(ipl))

# failed lookups: no country (a flag is created at merge time, rows are NOT dropped)
ipl.loc[~ipl["lookup_ok"], ["country", "country_code"]] = pd.NA

# proxy / hosting -> 0/1 (stays NaN when the API did not return them)
for c in ["proxy", "hosting"]:
    ipl[c] = (ipl[c].astype(str).str.lower()
              .map({"true": 1, "false": 0, "1": 1, "0": 0, "1.0": 1, "0.0": 0}))
log("api", "ip_lookup", "converted proxy and hosting to 0/1; left NaN when missing",
    "imputation (if any) must be fit after the train/test split, inside the pipeline", int(ipl[["proxy", "hosting"]].isna().any(axis=1).sum()))

n_failed = int((~ipl["lookup_ok"]).sum())
log("api", "ip_lookup", "counted failed lookups (private/reserved ranges, errors)",
    "they become country='Unknown' with ip_lookup_failed=1 instead of being dropped", n_failed)

ipl["country"] = ipl["country"].astype("object")

### 2.1 Merge the lookup onto the transactions

In [ ]:
lookup_for_merge = (ipl[["ip", "country", "country_code", "lookup_ok", "proxy", "hosting"]]
                    .rename(columns={"country": "ip_country_api", "country_code": "ip_country_code_api",
                                     "proxy": "is_proxy", "hosting": "is_hosting"}))

df = df.merge(lookup_for_merge, on="ip", how="left", validate="m:1")
assert len(df) == n_primary, "merge changed the number of rows"

lookup_ok = df["lookup_ok"].eq(True).fillna(False).astype(bool)
df["ip_lookup_failed"] = (~lookup_ok).astype(int)
df["ip_country_api"] = df["ip_country_api"].fillna("Unknown")
df = df.drop(columns="lookup_ok")

log("merge", "api", "left-merged ip_lookup on ip; failed or missing lookups -> 'Unknown' + ip_lookup_failed=1",
    "keeps every transaction and lets the model learn from 'unknown location'", int(df["ip_lookup_failed"].sum()))
print("lookup failure rate: %.2f%%" % (100 * df["ip_lookup_failed"].mean()))
df["ip_country_api"].value_counts().head(10)

### 2.2 Cross-check API vs static lookup, then choose the country column
Compared only where **both** sources returned a country. Remember: transactions are from 2015 and the API answers with today's geolocation, so some disagreement is expected. This match rate goes in the report.

In [ ]:
both = (df["ip_country_api"] != "Unknown") & (df["ip_country_static"] != "Unknown")
agree = df["ip_country_api"].str.casefold().eq(df["ip_country_static"].str.casefold())
if both.any():
    print(f"rows with both countries: {int(both.sum()):,} | agreement: {agree[both].mean():.1%}")
    display(df.loc[both & ~agree, ["ip_country_static", "ip_country_api"]].value_counts().head(15).to_frame("rows"))
else:
    print("No rows have both countries yet -> cannot cross-check.")
log("crosscheck", "api_vs_static", "compared API country with static-lookup country where both exist",
    "static lookup is a historical cross-check for the live API (2015 data vs today's geolocation)", int(both.sum()))

# the column that feeds the CPI join (team decision: see COUNTRY_SOURCE in the setup cell)
df["ip_country"] = df["ip_country_api"] if COUNTRY_SOURCE == "api" else df["ip_country_static"]
log("merge", "country", f"set ip_country from the {COUNTRY_SOURCE} source",
    "one agreed country column is needed for the CPI join; the other source stays as its own column", len(df))

**Data-quality note (API):** _TODO (team): failure rate, which IP ranges failed, missing fields, and the limitation that 2015 IPs are resolved with today's geolocation data._

## 3. Scraped data: Wikipedia CPI
### 3.1 Load and clean
Expected raw file: one country column and either (a) `cpi_score` + `year` columns, or (b) one column per year (`2015`, `2016`, ... or `score_2015`, `score_2016`, ... as saved by `02_collection`). **Adjust only the config line below if the scraper's column names differ.**

In [ ]:
cpi_raw = pd.read_csv(CPI_CSV) if CPI_CSV.exists() else pd.DataFrame(columns=["country", "cpi_score", "year"])
if not CPI_CSV.exists():
    print("WARNING: cpi_raw_table.csv not found -> empty placeholder, country_risk_index will be missing for all rows.")

cpi = cpi_raw.copy()
cpi.columns = [str(c).strip().lower() for c in cpi.columns]

# --- strip footnote markers like [a], [12] and odd whitespace from every text column ---
def clean_text(s):
    s = re.sub(r"\[[^\]]*\]", "", str(s))
    return s.replace("\xa0", " ").strip()

for c in cpi.select_dtypes(include="object").columns:
    cpi[c] = cpi[c].map(clean_text)
log("scrape", "cpi", "removed footnote markers ([a], [1]...) and non-breaking spaces",
    "scraped HTML text carries footnote noise that breaks joins and numeric parsing", len(cpi))

# --- find the country column and reshape wide -> long if needed ---
country_candidates = [c for c in cpi.columns if "country" in c or "nation" in c]
country_col = country_candidates[0] if country_candidates else cpi.columns[0]
year_cols = [c for c in cpi.columns if re.fullmatch(r"(score_)?\d{4}", c)]   # 02 saves score_<year>

if year_cols:
    cpi = cpi.melt(id_vars=[country_col], value_vars=year_cols, var_name="year", value_name="cpi_score")
    cpi["year"] = cpi["year"].str.replace("score_", "", regex=False)   # score_2015 -> 2015
else:
    cpi = cpi.rename(columns={"score": "cpi_score"})
cpi = cpi.rename(columns={country_col: "country"})[["country", "cpi_score", "year"]]

# --- types ---
cpi["year"] = pd.to_numeric(cpi["year"], errors="coerce")
score_before = cpi["cpi_score"].notna().sum()
cpi["cpi_score"] = pd.to_numeric(cpi["cpi_score"], errors="coerce")
log("scrape", "cpi", "converted cpi_score and year to numeric (errors -> NaN)",
    "scores were scraped as text", int(score_before - cpi["cpi_score"].notna().sum()))
cpi = cpi.dropna(subset=["country", "cpi_score", "year"])
cpi["year"] = cpi["year"].astype(int)
cpi.head()

In [ ]:
# --- keep one year: CPI_YEAR, or the closest available year to it ---
if len(cpi):
    avail = np.sort(cpi["year"].unique())
    year_used = int(avail[np.abs(avail - CPI_YEAR).argmin()])
    if year_used != CPI_YEAR:
        print(f"CPI {CPI_YEAR} not available, using closest year: {year_used}")
    cpi = cpi[cpi["year"] == year_used].drop_duplicates("country").reset_index(drop=True)
    log("scrape", "cpi", f"kept CPI year {year_used} only (one row per country)",
        "transactions are from 2015, so the closest CPI year avoids time mismatch", len(cpi))
print(cpi.shape)
cpi.describe(include="all").T

### 3.2 Country-name mapping (ip-api names <-> Wikipedia names)
Both sides are normalized (accents, case, punctuation, leading "the") and known aliases are collapsed to one canonical key. **The alias list is only a starting point: check the unmatched table below and extend it.** Unmatched countries are flagged, never guessed.

In [ ]:
ALIASES = {
    "czechia": "czech republic",
    "russian federation": "russia",
    "burma": "myanmar",
    "swaziland": "eswatini",
    "macedonia": "north macedonia",
    "turkiye": "turkey",
    "viet nam": "vietnam",
    "united states of america": "united states",
    "cote d ivoire": "ivory coast",
    "cabo verde": "cape verde",
    "dr congo": "democratic republic of the congo",
    "republic of korea": "south korea",
    "korea republic of": "south korea",
    "lao pdr": "laos",
    "brunei darussalam": "brunei",
    "syrian arab republic": "syria",
    "iran islamic republic of": "iran",
    "bolivia plurinational state of": "bolivia",
    "venezuela bolivarian republic of": "venezuela",
    "tanzania united republic of": "tanzania",
    "moldova republic of": "moldova",
    "timor leste": "east timor",
}

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    s = s.lower().replace("&", "and")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return re.sub(r"^the ", "", s)

def canon(s):
    k = norm(s)
    return ALIASES.get(k, k)

country_counts = (df["ip_country"].value_counts()
                  .rename_axis("ip_country").reset_index(name="n_transactions"))
country_counts["key"] = country_counts["ip_country"].map(canon)

cpi_keyed = cpi.assign(key=cpi["country"].map(canon)).drop_duplicates("key")
country_map = country_counts.merge(
    cpi_keyed[["key", "country"]].rename(columns={"country": "cpi_country"}),
    on="key", how="left")

exact = country_map["ip_country"] == country_map["cpi_country"]
country_map["match_type"] = np.select(
    [country_map["ip_country"] == "Unknown", exact, country_map["cpi_country"].notna()],
    ["unknown_lookup", "exact", "alias_or_normalized"], default="unmatched")

known = country_map[country_map["ip_country"] != "Unknown"]
rate_countries = (known["match_type"] != "unmatched").mean() if len(known) else float("nan")
rate_tx = (known.loc[known["match_type"] != "unmatched", "n_transactions"].sum()
           / max(known["n_transactions"].sum(), 1))
print(f"match rate by country: {rate_countries:.1%}  |  weighted by transactions: {rate_tx:.1%}")
print(country_map["match_type"].value_counts())

print("\nUNMATCHED countries (largest first) - extend ALIASES for the important ones:")
display(country_map[country_map["match_type"] == "unmatched"]
        .sort_values("n_transactions", ascending=False)[["ip_country", "n_transactions"]].head(25))

### 3.3 Merge CPI onto the transactions

In [ ]:
df = df.merge(country_map[["ip_country", "cpi_country"]], on="ip_country", how="left", validate="m:1")
df = df.merge(cpi[["country", "cpi_score"]].rename(columns={"country": "cpi_country"}),
              on="cpi_country", how="left", validate="m:1")
assert len(df) == n_primary, "merge changed the number of rows"

# CPI: 0 = very corrupt, 100 = very clean. Flip it so that HIGHER = RISKIER.
df["country_risk_index"] = 100 - df["cpi_score"]
df["cpi_missing"] = df["country_risk_index"].isna().astype(int)
df = df.drop(columns=["cpi_country", "cpi_score"])

log("merge", "cpi", "joined CPI by mapped country; country_risk_index = 100 - CPI score; unmatched -> NaN + cpi_missing=1",
    "CPI is higher for cleaner countries, so it is flipped to read as risk; unmatched countries are flagged, not guessed",
    int(df["cpi_missing"].sum()))
print("rows without CPI: %.2f%%" % (100 * df["cpi_missing"].mean()))

**Data-quality note (scrape):** _TODO (team): match rate (from 3.2), which countries needed manual aliases, which stayed unmatched and why, CPI year used._

## 4. Validate the final table

In [ ]:
clean = df[[
    "user_id", "signup_time", "purchase_time", "purchase_value", "device_id",
    "source", "browser", "sex", "age", "ip_address_raw", "ip", "ip_int",
    "ip_country", "ip_country_api", "ip_country_static", "static_matched",
    "ip_country_code_api", "ip_lookup_failed", "is_proxy", "is_hosting",
    "country_risk_index", "cpi_missing", "class",
]].copy()

assert len(clean) == n_primary
assert clean["user_id"].is_unique
assert clean["class"].isin([0, 1]).all()
assert clean[["user_id", "signup_time", "purchase_time", "purchase_value", "class", "ip_country"]].notna().all().all()

print("shape:", clean.shape)
print("fraud rate: %.2f%%" % (100 * clean["class"].mean()))
display(clean.dtypes.to_frame("dtype").join(clean.isna().sum().to_frame("nulls")))
clean.head()

Remaining nulls are **expected and flagged**: `is_proxy`/`is_hosting` (if the API did not return them) and `country_risk_index` (see `cpi_missing`). Imputing them happens later, inside the modeling pipeline, after the train/test split.

## 5. Save: CSV, SQL database, cleaning log

In [ ]:
suffix = "_PLACEHOLDER" if (USING_PLACEHOLDER and COUNTRY_SOURCE == "api") else ""
out_csv = PROC / f"clean_merged{suffix}.csv"
clean.to_csv(out_csv, index=False)
print("saved:", out_csv)

# --- SQLite for now (same schema can be loaded into PostgreSQL later) ---
tables = {
    "raw_transactions": raw,
    "ip_lookup": ipl.drop(columns=["lookup_ok"]),
    "cpi": cpi,
    "country_map": country_map.drop(columns=["key"]),
    "clean_merged": clean,
}
if STATIC_AVAILABLE:
    tables["ip_ranges_static"] = static
with sqlite3.connect(DB_PATH) as con:
    for name, t in tables.items():
        t.to_sql(name, con, if_exists="replace", index=False)
        print(f"table {name}: {len(t)} rows")

log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(PROC / "cleaning_log.csv", index=False)
log_df

### 5.1 Queries the dashboard will need
These run on SQLite. In PostgreSQL replace `strftime('%H', purchase_time)` with `EXTRACT(HOUR FROM purchase_time)`.

In [ ]:
queries = {
    "fraud_by_country": """
        SELECT ip_country, COUNT(*) AS n, SUM("class") AS frauds,
               ROUND(100.0 * AVG("class"), 2) AS fraud_rate_pct
        FROM clean_merged GROUP BY ip_country HAVING COUNT(*) >= 50
        ORDER BY fraud_rate_pct DESC LIMIT 15""",
    "fraud_by_browser": """
        SELECT browser, COUNT(*) AS n, ROUND(100.0 * AVG("class"), 2) AS fraud_rate_pct
        FROM clean_merged GROUP BY browser ORDER BY fraud_rate_pct DESC""",
    "fraud_by_source": """
        SELECT source, COUNT(*) AS n, ROUND(100.0 * AVG("class"), 2) AS fraud_rate_pct
        FROM clean_merged GROUP BY source ORDER BY fraud_rate_pct DESC""",
    "fraud_by_hour": """
        SELECT CAST(strftime('%H', purchase_time) AS INTEGER) AS hour, COUNT(*) AS n,
               ROUND(100.0 * AVG("class"), 2) AS fraud_rate_pct
        FROM clean_merged GROUP BY hour ORDER BY hour""",
}
with sqlite3.connect(DB_PATH) as con:
    for name, q in queries.items():
        print("\n==", name)
        display(pd.read_sql(q, con))

## 6. Summary for the report (fill in)
| Item | Result |
|---|---|
| Rows in / out | _TODO_ |
| Nulls in primary data | _TODO_ |
| Duplicates / impossible values removed | _TODO_ |
| API lookup failure rate | _TODO_ |
| CPI country match rate | _TODO_ |
| Final table | `data/processed/clean_merged.csv` |

**Next:** `04_eda.ipynb` and `05_features.ipynb` (features, stratified split, then encoders/scalers fit on train only).